# 

Practical Application III: Comparing Classifiers

**Overview**: In this practical application, your goal is to compare the performance of the classifiers we encountered in this section, namely K Nearest Neighbor, Logistic Regression, Decision Trees, and Support Vector Machines.  We will utilize a dataset related to marketing bank products over the telephone.  

### Getting Started

Our dataset comes from the UCI Machine Learning repository [link](https://archive.ics.uci.edu/ml/datasets/bank+marketing).  The data is from a Portugese banking institution and is a collection of the results of multiple marketing campaigns.  We will make use of the article accompanying the dataset [here](CRISP-DM-BANK.pdf) for more information on the data and features.



### Problem 1: Understanding the Data

To gain a better understanding of the data, please read the information provided in the UCI link above, and examine the **Materials and Methods** section of the paper.  How many marketing campaigns does this data represent?

According to the dataset documentation and accompanying paper, the data was collected from 17 marketing campaigns conducted by a Portuguese banking institution between May 2008 and November 2010.

### Data Preparation

The dataset was loaded into a pandas DataFrame. Several preprocessing steps were performed before modeling:

- The `duration` feature was removed because it causes data leakage and is only known after the marketing call has ended.
- The target variable `y` was converted from categorical values (`yes` and `no`) to binary values (`1` and `0`).
- The value `999` in `pdays` was replaced with `-1` to indicate that the customer had not been previously contacted.

### Problem 2: Read in the Data

Use pandas to read in the dataset `bank-additional-full.csv` and assign to a meaningful variable name.

In [ ]:
import pandas as pd

bank_data = pd.read_csv("bank-additional-full.csv", sep=';')

# Preview cleaned data
bank_data.head()


### Problem 3: Data Cleaning
The dataset does not contain traditional missing values. However, several categorical features contain the value `"unknown"`, which represents unavailable information. To prepare the data for modeling:

- The `duration` feature was removed because it introduces data leakage.
- The target variable `y` was converted from `"yes"`/`"no"` to `1`/`0`.
- The value `999` in `pdays` was replaced with `-1` to indicate that the customer had not been previously contacted.
- Data types and missing values were verified before modeling.

In [ ]:
# Remove leakage feature
bank_data = bank_data.drop(columns=['duration'])

# Encode target
bank_data['y'] = bank_data['y'].map({'yes': 1, 'no': 0})


# Replace special pdays value
bank_data['pdays'] = bank_data['pdays'].replace(999, -1)

# Check for missing values
print(bank_data.isnull().sum())

# Verify data types
bank_data.info()


### Problem 4: Business Objective
After examining the description and data, your goal now is to clearly state the *Business Objective* of the task.  State the objective below.

The objective of this project is to develop a machine learning model that predicts whether a bank customer will subscribe to a term deposit after a marketing phone call.

By accurately identifying customers who are more likely to subscribe, the bank can:

- Improve marketing efficiency
- Reduce campaign costs
- Increase conversion rates
- Focus resources on high-potential customers

This predictive model can help the bank target future marketing campaigns more effectively and improve the overall return on marketing investments.

### Problem 5: Feature Engineering

To prepare the data for machine learning, the bank information features were selected as predictors and the target variable `y` was used as the response variable.

Categorical features were encoded using One-Hot Encoding so that machine learning algorithms could process them. Numerical features were passed through without modification. A `ColumnTransformer` was used to apply the appropriate transformations to each feature type.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Select features and target
features = ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan']

X = bank_data[features]
y = bank_data['y']

# Identify categorical and numerical columns
cat_cols = X.select_dtypes(include='object').columns
num_cols = X.select_dtypes(exclude='object').columns

# Apply One-Hot Encoding to categorical variables
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('num', 'passthrough', num_cols)
])

# Display selected columns
print("Categorical Features:", list(cat_cols))
print("Numerical Features:", list(num_cols))

### Problem 6: Train/Test Split

The dataset was split into training and testing sets using an 80/20 split. Stratified sampling was used to preserve the original class distribution of the target variable in both datasets. A random state of 42 was specified to ensure reproducibility of the results.

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Split data first
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print("Training Set Distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting Set Distribution:")
print(y_test.value_counts(normalize=True))

### Problem 7: Target Class Distribution

In [ ]:
import matplotlib.pyplot as plt

y.value_counts().plot(kind='bar')

plt.title('Target Variable Distribution')
plt.xlabel('Subscribed (0=No, 1=Yes)')
plt.ylabel('Count')
plt.show()

The target variable is highly imbalanced, with approximately 89% of customers not subscribing and only 11% subscribing. This imbalance motivates the use of Recall and F1 Score in addition to Accuracy when evaluating model performance. As a result, class weights and F1-based model evaluation were used throughout this analysis.

### Problem 8: A Baseline Model

The dataset is highly imbalanced. Approximately 88.7% of customers did not subscribe to a term deposit, while only about 11.3% subscribed.

Therefore, a naïve classifier that always predicts the majority class ("No") would achieve a baseline accuracy of approximately 88.7%.

Any machine learning model should outperform this baseline while also improving recall and F1 score for the minority class.

In [ ]:
y.value_counts(normalize=True)

### Problem 9: A Simple Model

A Logistic Regression model was trained using the encoded training data. Because the dataset is imbalanced, class weights were balanced to give more importance to the minority class during training.


In [ ]:
from sklearn.linear_model import LogisticRegression

# Encode data
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

# Train Logistic Regression model
log_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

log_model.fit(X_train_encoded, y_train)

# Generate predictions
log_pred = log_model.predict(X_test_encoded)


### Problem 10: Score the Model

The performance of the Logistic Regression model was evaluated using accuracy on the test set. Accuracy measures the proportion of correct predictions made by the model.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, recall_score

def evaluate(name, y_test, y_pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred)
    }
log_accuracy = accuracy_score(y_test, log_pred)
print("Logistic Regression Accuracy:", log_accuracy)    

### Problem 11: Model Comparisons
Four classifiers were evaluated: Logistic Regression, K-Nearest Neighbors (KNN), Decision Tree, and Support Vector Machine (SVM). Performance was compared using Accuracy, Recall, and F1 Score.


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, recall_score
import pandas as pd

def evaluate(name, y_true, y_pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "F1 Score": f1_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred)
    }

results = []

# Logistic Regression from Problem 9
results.append(evaluate("Logistic Regression", y_test, log_pred))

results_df = pd.DataFrame(results)

print(results_df)

### Problem 12: Improving the Model
To improve model performance, hyperparameter tuning was performed using GridSearchCV. Since the dataset is imbalanced, F1 Score was used as the optimization metric instead of accuracy. The models evaluated were Logistic Regression, KNN, Decision Tree, and SVM.

The best hyperparameters were selected using GridSearchCV with cross-validation. The tuned models were then evaluated on the test set using Accuracy, Recall, and F1 Score.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC

# Logistic Regression
log_grid = GridSearchCV(
    LogisticRegression(max_iter=1000),
    {'C': [0.1, 1, 10], 'class_weight': ['balanced']},
    cv=2,
    scoring='f1'
)

# KNN
knn_grid = GridSearchCV(
    KNeighborsClassifier(),
    {'n_neighbors': [3, 5, 7], 'weights': ['uniform', 'distance']},
    cv=2,
    scoring='f1'
)

# Decision Tree
tree_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    {'max_depth': [3, 5, 10], 'class_weight': ['balanced']},
    cv=2,
    scoring='f1'
)

# SVM
svm_grid = GridSearchCV(
    LinearSVC(),
    {'C': [0.1, 1, 10]},
    cv=2,
    scoring='f1'
)

# Train models
log_grid.fit(X_train_encoded, y_train)
knn_grid.fit(X_train_encoded, y_train)
tree_grid.fit(X_train_encoded, y_train)
svm_grid.fit(X_train_encoded, y_train)

# Predictions
log_pred = log_grid.predict(X_test_encoded)
knn_pred = knn_grid.predict(X_test_encoded)
tree_pred = tree_grid.predict(X_test_encoded)
svm_pred = svm_grid.predict(X_test_encoded)

# Compare tuned models
results = []

results.append(evaluate("Logistic Regression (Tuned)", y_test, log_pred))
results.append(evaluate("KNN (Tuned)", y_test, knn_pred))
results.append(evaluate("Decision Tree (Tuned)", y_test, tree_pred))
results.append(evaluate("SVM (Tuned)", y_test, svm_pred))

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="F1 Score", ascending=False)

print(results_df)

print("\nBest Model:", results_df.iloc[0]["Model"])

### Problem 13: Findings

The tuned models were compared using Accuracy, Recall, and F1 Score. Since the dataset is imbalanced, F1 Score was used as the primary evaluation metric.

The best-performing model was Decision Tree (Tuned) which achieved the highest F1 Score. This suggests that it provides the best balance between precision and recall when identifying customers likely to subscribe to a term deposit.

These results indicate that machine learning can help the bank target customers more effectively, potentially reducing marketing costs and improving campaign conversion rates.

### Problem 14: Visualization 
The chart compares Accuracy, Recall, and F1 Score across the tuned classifiers. The Decision Tree achieved the highest F1 Score, indicating the best balance between precision and recall for predicting term deposit subscriptions. Although several models achieved high accuracy, F1 Score was considered the primary metric because of the class imbalance present in the dataset.

In [ ]:
results_df.plot(
    x='Model',
    y=['Accuracy', 'Recall', 'F1 Score'],
    kind='bar'
)

plt.title('Classifier Performance Comparison')
plt.ylabel('Score')
plt.xticks(rotation=20)
plt.show()

##### Questions